# Tutorial 1 — Export one SDShiftGELU component

This self-contained notebook copies the PyTorch `ISDShiftGELU` arithmetic directly into the notebook. It does not import the block from LW-DETR. The exported ONNX still contains ordinary arithmetic operators; Aidge recognizes and fuses that pattern in the companion import notebook.

Run this notebook with the **`lwdetr2`** environment. Artifacts are written under `experiments/sd_components/`.

In [17]:
from collections import Counter
from pathlib import Path
import json
import time

import numpy as np
import onnx
import onnxruntime as ort
from onnx import shape_inference
from polygraphy.backend.onnx.loader import fold_constants
import torch
from torch import nn
from torch.autograd import Function

EXP = Path.cwd() if Path.cwd().name == 'experiments' else Path.cwd() / 'experiments'
ARTIFACTS = EXP / 'sd_components'
MODEL_DIR = ARTIFACTS / 'models'
DATA_DIR = ARTIFACTS / 'data'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(11)
np.random.seed(11)
WARMUP_RUNS = 10
TIMED_RUNS = 200
print('models:', MODEL_DIR.resolve())
print('data:  ', DATA_DIR.resolve())

models: /home/tl286648/dev/model/LW-DETR/experiments/sd_components/models
data:   /home/tl286648/dev/model/LW-DETR/experiments/sd_components/data


## 1. Self-contained PyTorch implementation

These STE helpers and `ISDShiftGELU` are copied from the LW-DETR fake-quant implementation. During inference their forward methods are exactly `floor` and `round`; the custom backward methods matter only during training.

In [18]:
class floor_ste(Function):
    @staticmethod
    def forward(ctx, x):
        return torch.floor(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


class round_ste(Function):
    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


class ISDShiftGELU(nn.Module):
    def __init__(self, output_bit=8, n=23, temp=1.702, channel_len=None):
        super().__init__()
        self.output_bit = output_bit
        self.n = n
        self.m = 31
        scale_shape = channel_len if channel_len is not None else 1
        self.register_buffer('act_scaling_factor', torch.zeros(scale_shape))
        self.register_buffer(
            'sigmoid_scaling_factor',
            torch.tensor([1 / 2.0 ** (self.output_bit - 1)]),
        )
        self.temp = temp

    def fix(self):
        pass

    def unfix(self):
        pass

    def int_exp_shift(self, x_int, scaling_factor):
        scaled_floor = floor_ste.apply(x_int / 2)
        scaled_exp_floor = floor_ste.apply(x_int / (2.0**4))
        x_int = x_int + scaled_floor - scaled_exp_floor
        with torch.no_grad():
            x0_int = torch.ceil(1.0 / scaling_factor)
        min_x_int = torch.clamp(self.n * (-x0_int), min=-(2.0**self.m))
        x_int = torch.maximum(x_int, min_x_int)
        q = floor_ste.apply(x_int / (-x0_int))
        r = -(x_int - q * (-x0_int))
        exp_int = floor_ste.apply((-r) / 2)
        exp_int += x0_int
        exp_int = floor_ste.apply(exp_int * 2.0 ** (self.n - q))
        scaling_factor = scaling_factor / 2.0**self.n
        return exp_int, scaling_factor

    def forward(self, x, act_scaling_factor=None):
        pre_x_int = round_ste.apply(x / act_scaling_factor)
        scaling_factor_sig = act_scaling_factor * self.temp
        x_int_max = torch.clamp(pre_x_int, min=0)
        x_int = pre_x_int - x_int_max
        exp_int, _ = self.int_exp_shift(x_int, scaling_factor_sig)
        exp_int_max, _ = self.int_exp_shift(-x_int_max, scaling_factor_sig)
        exp_int_sum = exp_int + exp_int_max
        exp_int_sum = torch.clamp(exp_int_sum, min=1, max=2.0**self.m - 1)
        factor = floor_ste.apply((2.0**self.m - 1) / exp_int_sum)
        sigmoid_int = floor_ste.apply(
            exp_int * factor / (2.0 ** (self.m - self.output_bit + 1))
        )
        x_int_out = pre_x_int * sigmoid_int
        out_scaling_factor = act_scaling_factor * self.sigmoid_scaling_factor
        self.act_scaling_factor = out_scaling_factor
        out = x_int_out * out_scaling_factor
        return out, out_scaling_factor

## 2. Build the component and PyTorch reference

The leading `round` is a visible model-boundary operation and remains outside the fused SDShiftGELU pattern. The same deterministic input and PyTorch reference are saved for the Aidge notebook.

In [19]:
class OneSDShiftGELU(nn.Module):
    def __init__(self, input_scale=0.05):
        super().__init__()
        self.activation = ISDShiftGELU(output_bit=8, n=23, temp=1.702)
        self.register_buffer(
            'input_scale', torch.tensor([input_scale], dtype=torch.float32)
        )

    def forward(self, x):
        output, _ = self.activation(x, self.input_scale)
        return output


model = OneSDShiftGELU().eval()
integer_input = torch.round(
    torch.linspace(-40, 40, 24, dtype=torch.float32)
).reshape(2, 3, 4).to(torch.int32)
# ONNX fake-quant export uses dequantized float carriers, reconstructed
# directly from the integer codes. No float fixture is saved.
input_tensor = integer_input.to(torch.float32) * model.input_scale
def average_latency_ms(function, warmup=WARMUP_RUNS, repeats=TIMED_RUNS):
    for _ in range(warmup):
        function()
    start = time.perf_counter()
    for _ in range(repeats):
        function()
    return (time.perf_counter() - start) * 1000.0 / repeats


with torch.no_grad():
    pytorch_output = model(input_tensor)
    pytorch_latency_ms = average_latency_ms(lambda: model(input_tensor))
output_scale = model.input_scale * model.activation.sigmoid_scaling_factor
pytorch_integer_output = torch.round(pytorch_output / output_scale).to(torch.int32)

np.save(DATA_DIR / 'sdshiftgelu_integer_input.npy', integer_input.numpy())
np.save(DATA_DIR / 'sdshiftgelu_pytorch_integer_output.npy', pytorch_integer_output.numpy())
with open(DATA_DIR / 'sdshiftgelu_metadata.json', 'w') as file:
    json.dump({
        'input_scale': float(model.input_scale.item()),
        'output_scale': float(output_scale.item()),
        'pytorch_latency_ms': pytorch_latency_ms,
    }, file, indent=2)
print('input shape:', tuple(input_tensor.shape))
print('PyTorch output:', pytorch_output.numpy())
print('Integer input codes:', integer_input.numpy())
print('Integer output codes:', pytorch_integer_output.numpy())
print(f'PyTorch average latency: {pytorch_latency_ms:.6f} ms '
      f'({TIMED_RUNS} runs after {WARMUP_RUNS} warm-ups)')

input shape: (2, 3, 4)
PyTorch output: [[[-0.0625     -0.07226562 -0.09023438 -0.10546875]
  [-0.13203125 -0.13476562 -0.15585938 -0.15625   ]
  [-0.15468751 -0.13359375 -0.09375    -0.046875  ]]

 [[ 0.0515625   0.140625    0.2953125   0.421875  ]
  [ 0.6375      0.75703126  0.97031254  1.1578125 ]
  [ 1.3359375   1.546875    1.734375    1.875     ]]]
Integer input codes: [[[-40 -37 -33 -30]
  [-26 -23 -19 -16]
  [-12  -9  -5  -2]]

 [[  2   5   9  12]
  [ 16  19  23  26]
  [ 30  33  37  40]]]
Integer output codes: [[[-160 -185 -231 -270]
  [-338 -345 -399 -400]
  [-396 -342 -240 -120]]

 [[ 132  360  756 1080]
  [1632 1938 2484 2964]
  [3420 3960 4440 4800]]]
PyTorch average latency: 0.688731 ms (200 runs after 10 warm-ups)


## 3. Export, simplify, and verify with ONNX Runtime

The simplified ONNX is still a portable graph of standard operators—there is no custom SDShiftGELU node yet. Fusion happens only after Aidge imports it.

In [20]:
raw_path = MODEL_DIR / 'sdshiftgelu_only.onnx'
sim_path = MODEL_DIR / 'sdshiftgelu_only.sim.onnx'

torch.onnx.export(
    model,
    input_tensor,
    raw_path,
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
)
raw_model = onnx.load(raw_path)
onnx.checker.check_model(raw_model)
sim_model = fold_constants(
    raw_model, allow_onnxruntime_shape_inference=False
)
sim_model = shape_inference.infer_shapes(sim_model)
onnx.checker.check_model(sim_model)
onnx.save(sim_model, sim_path)

session = ort.InferenceSession(str(sim_path), providers=['CPUExecutionProvider'])
onnx_output = session.run(None, {'input': input_tensor.numpy()})[0]
onnx_latency_ms = average_latency_ms(
    lambda: session.run(None, {'input': input_tensor.numpy()})
)
print('ONNX operators:', dict(sorted(Counter(n.op_type for n in sim_model.graph.node).items())))
print('PyTorch vs ONNX max abs:', float(np.max(np.abs(pytorch_output.numpy() - onnx_output))))
print(f'ONNX Runtime average latency: {onnx_latency_ms:.6f} ms')
np.testing.assert_allclose(onnx_output, pytorch_output.numpy(), rtol=0, atol=0)
print('saved:', sim_path.resolve())

[I] Folding Constants | Pass 1
[I]     Total Nodes | Original:    79, After Folding:    55 |    24 Nodes Folded
[I] Folding Constants | Pass 2
[I]     Total Nodes | Original:    55, After Folding:    55 |     0 Nodes Folded
ONNX operators: {'Add': 5, 'Clip': 2, 'Div': 10, 'Floor': 12, 'Max': 2, 'Mul': 8, 'Neg': 5, 'Pow': 2, 'Reciprocal': 1, 'Round': 1, 'Sub': 7}
PyTorch vs ONNX max abs: 0.0
ONNX Runtime average latency: 0.055146 ms
saved: /home/tl286648/dev/model/LW-DETR/experiments/sd_components/models/sdshiftgelu_only.sim.onnx


## Next step

Run `import_sd_components_aidge.ipynb` with the `aidge` environment. It loads this ONNX, checks that exactly one native `SDShiftGELU` is produced, and compares PyTorch, ONNX Runtime, and Aidge outputs.